# Experiment 2 — Jev Query Scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nipundavid/ai-experiments/blob/main/jev/src/experiment_2_score.ipynb)

This notebook reproduces the `experiment_2_score.py` workflow: it asks Jev to score a query's complexity with an ordered rubric and stores the result in LangGraph state.

> Set `TYPESAFE_API_KEY` before running this notebook.

## Overview

This experiment tests when a numeric score is more informative than a binary or category label.

The workflow asks a single `Score` question:

- How complex is this query to answer accurately?

It then stores the score and rubric legend in a typed state.

In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

if 'TYPESAFE_API_KEY' not in os.environ:
    raise RuntimeError('Set the TYPESAFE_API_KEY environment variable before running this notebook.')

In [ ]:
from typing import TypedDict

from langchain_typesafe import Score, TypeSafeClassifier
from langgraph.graph import END, START, StateGraph

classifier = TypeSafeClassifier(api_key=os.environ['TYPESAFE_API_KEY'])

class State(TypedDict):
    query: str
    complexity: float
    rubric: list[str]


def score_query(state: State):
    response = classifier.invoke(
        {
            'state': state['query'],
            'questions': {
                'complexity': Score(
                    instructions='How complex is this query to answer accurately?',
                    criteria=[
                        'Can be answered directly in one or two sentences.',
                        'Needs a short explanation or a small number of steps.',
                        'Needs substantial reasoning, multiple steps, or external context.',
                    ],
                )
            },
        }
    )

    answer = response.scores['complexity']
    return {'complexity': answer.score, 'rubric': answer.legend}


graph = StateGraph(State)
graph.add_node('score_query', score_query)
graph.add_edge(START, 'score_query')
graph.add_edge('score_query', END)
app = graph.compile()

In [ ]:
query = 'Explain how hybrid search combines keyword and vector search.'
result = app.invoke({'query': query})

print('\n' + '=' * 70)
print('FINAL LANGGRAPH STATE')
print('=' * 70)
print(f"Query: {result['query']}")
print(f"Complexity score: {result['complexity']:.2f}")
print(f"Rubric: {result['rubric']}")

## Result interpretation

This notebook demonstrates that Jev can return a scored, ordered answer rather than a binary label. That makes it useful for complexity estimation, prioritization, and confidence-style measurements when a numeric judgment is more informative than a simple yes/no decision.